# A 股日常入口 · 阶段 4

阶段 1 是工具和看图。阶段 2 加上回测、信号表、清单和事件日历。阶段 3 记手工成交并和信号对照。这一页在此之上扣研究用费用，给出相对买入持有和沪深 300 的超额，并用可配置日期把区间分成 train / test。

大约每月 3% 仍是策略升级的**方向**，不是这一页的承诺，也不是示例策略的验收。不要把短均线改密去凑这个数字。

试用记录（产品预期，只描述过滤后的那次示例，不是核定结论，更不是交易指令）：工具链能用；过滤有帮助，但粗算月收益仍大约在 -0.09% 到 -0.26%，离大约 3%/月还有约 3 个百分点。

**【跟单视图锁定为 next_open】** 第一行才是跟单假设。收盘成交只并排作对照。

止损默认看收盘价，不是盘中触发。盘中止损是以后的假设，这一页没有。可选的「次日开盘止损」只把止损的离场改成下一根开盘。

请从仓库根目录启动 Jupyter（见 README）。

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import sys

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    for parent in ROOT.parents:
        if (parent / "pyproject.toml").exists():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backtest import FOLLOW_LABEL, compare_fills, fill_assumption_table, run_backtest
from data import (
    available_universes,
    default_cache_dir,
    get_bars,
    get_csi300,
    load_events,
    universe_names,
    universe_note,
    universe_symbols,
)
from journal import apply_cost_schedule, compare_actual, format_report, save_fills, simulate_follow_fills
from orders import PAUSE_LABEL, export_order_list
from strategy import (
    DualMovingAverage,
    EventGuard,
    MeanReversion,
    apply_diagnostics,
    available_strategies,
    event_switch_samples,
    mean_reversion_sensitivity,
    position_snapshot,
    signal_board,
)
from viz import plot_signals, save_chart

pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

CACHE_DIR = default_cache_dir()
print("仓库根目录:", ROOT)
print("行情缓存:", CACHE_DIR)
print("已注册策略:", ", ".join(available_strategies()))
print("可选样本:", ", ".join(available_universes()))

## 参数

`SAMPLE = "demo"` 是平安银行和贵州茅台，缓存里通常已有，可以离线。改成 `banks`、`consumer` 或 `liquid` 会用更宽的静态样本（银行、消费），第一次需要联网。那不是实时指数成分。

`EVENT_MODE` 可改成 `block_entries` 或 `reduce`。默认 `off`。页末有一段不拉行情的开关样本。

止损用收盘价。`PAUSE_AFTER_LOSSES` 默认 3：连续亏损达到这个数，或短持翻脸的卖出出现至少 2 次，清单写「建议暂停加仓」。


In [ ]:
SAMPLE = "demo"  # demo / banks / consumer / liquid
NAMES = universe_names(SAMPLE)
SYMBOLS = universe_symbols(SAMPLE)
END = date.today()
START = END - timedelta(days=365 * 5)
ADJUST = "qfq"
REFRESH = False
OFFLINE = False
EVENT_MODE = "off"          # off / block_entries / reduce
FETCH_EVENTS = False
ORDER_LOOKBACK_DAYS = 90
PAUSE_AFTER_LOSSES = 3
EQUITY = 100_000            # 演示成交的参考资金，元
SPLIT_DATE = "2024-01-01"   # 这一天及之后是 test，之前是 train
COMMISSION_RATE = 0.00025   # 研究用费率，不是券商价目
STAMP_TAX_RATE = 0.0005     # 默认只计在卖出
MIN_COMMISSION = 5.0
OUTPUT = ROOT / "output"

print(f"样本 {SAMPLE}：{universe_note(SAMPLE)}")
print("代码:", ", ".join(f"{code} {NAMES[code]}" for code in SYMBOLS))
print("更宽的 liquid:", ", ".join(universe_symbols("liquid")))
print(f"区间 {START} ~ {END} · 复权 {ADJUST or '不复权'} · 事件约束 {EVENT_MODE}")
print(f"【跟单视图锁定为 next_open】分段 {SPLIT_DATE} · 研究用佣金 {COMMISSION_RATE} · 卖出印花税 {STAMP_TAX_RATE}")

## 拉数

同一区间第二次运行命中 `.cache/bars/`。前复权价会在除权后被追溯调整，需要对齐最新价时把上面的 `REFRESH` 改成 `True`。


In [ ]:
books = {}
for symbol in SYMBOLS:
    bars = get_bars(
        symbol,
        START,
        END,
        adjust=ADJUST,
        refresh=REFRESH,
        offline=OFFLINE,
        cache_dir=CACHE_DIR,
    )
    code = str(bars["symbol"].iloc[0])
    books[code] = bars
    hit = "缓存" if bars.attrs.get("cache_hit") else "网络"
    print(
        f"{code} {NAMES.get(code, '')}  {len(bars)} 根  "
        f"{bars['date'].min():%Y-%m-%d}~{bars['date'].max():%Y-%m-%d}  "
        f"{bars.attrs.get('source')} ({hit})"
    )


## 策略

默认仍是 10 日 / 30 日，但不是裸交叉，也不是把短均线再改短：

- 连续 2 个交易日确认
- 最短持有 5 个交易日；止损可以提前离场
- 收盘价要站上 60 日趋势均线
- ADX 近似不低于 20
- 建议仓位 50%
- 止损 8%，**用收盘价相对入场收盘价**。盘中触发不在这里

事件日约束默认关闭。图上仍列出本地事件。


In [ ]:
strategy = DualMovingAverage()
print(
    f"{strategy.name}: 确认 {strategy.confirm_days} 日, 最短持有 {strategy.min_hold_days} 日, "
    f"趋势 {strategy.trend_window}, ADX {strategy.adx_window}/{strategy.adx_min}, "
    f"建议仓位 {strategy.target_weight:.0%}, 止损 {strategy.stop_loss}（收盘触发）, 止盈 {strategy.take_profit}"
)

events = load_events(start=START, end=END, fetch=FETCH_EVENTS)
if events.attrs.get("fetch_error"):
    print("事件外源失败，已退回本地日历:", events.attrs["fetch_error"])
print(f"事件 {len(events)} 条 · {events.attrs.get('source')}")

guarded = {}
shown = {}
for code, bars in books.items():
    raw = strategy.run(bars)
    guarded[code] = EventGuard(events, mode=EVENT_MODE).apply(raw, bars)
    shown[code] = apply_diagnostics(guarded[code], bars)
    signals = shown[code].signals
    flips = int(signals["flag"].eq("短持翻脸").sum()) if not signals.empty else 0
    weight = float(shown[code].positions.iloc[-1]["weight"])
    print(f"{code} 信号 {len(signals)} 条, 其中短持翻脸标记 {flips} 条, 最新建议仓位 {weight:.0%}")


## 回测：跟单视图锁定为 next_open

**【跟单视图锁定为 next_open】** 第一行才是跟单假设。**回测未扣费；对账看成交日志**。收盘成交放在旁边，只说明「如果收盘就能成交」会差多少。回测本身没有滑点、佣金、印花税、涨跌停。费用记在后面的成交对照里。

第三行是「次日开盘止损」：止损仍在收盘触发，假定下一根开盘才卖出。其他买卖仍按收盘假设。粗算月收益不是每月 3% 的验收。

In [ ]:
tables = []
for code, bars in books.items():
    compared = compare_fills(bars, guarded[code])
    stop_fill = run_backtest(bars, guarded[code], fill="next_open_stop")
    table = fill_assumption_table(compared, stop_fill)
    tables.append(table)
assumptions = pd.concat(tables, ignore_index=True)
view = assumptions.copy()
for column in ("total_return", "max_drawdown", "rough_monthly_return"):
    view[column] = view[column].map(lambda value: f"{value:.2%}")
view["turnover"] = view["turnover"].map(lambda value: f"{float(value):.2f}")
print("【跟单视图锁定为 next_open】第一行才是跟单假设。")
print(FOLLOW_LABEL)
print("**回测未扣费；对账看成交日志**")
display(view[["symbol", "label", "follow", "total_return", "max_drawdown", "turnover", "rough_monthly_return"]])
print("粗算月收益不是每月 3% 的验收。示例不保证收益，也不靠缩短短均线去凑。")

## 信号表

最近一个交易日若没有新的交叉，表是空的。当前腿的持有天数、浮动盈亏、连亏和距上次交叉，看下面的持仓快照。持有不超过 5 个交易日就完成的来回，标记为「短持翻脸」。


In [ ]:
latest_day = max(frame["date"].max() for frame in books.values())
today = signal_board(shown, strategy_name=strategy.name, on=latest_day, bars=books)
window_start = END - timedelta(days=ORDER_LOOKBACK_DAYS)
window = signal_board(
    shown,
    strategy_name=strategy.name,
    start=window_start,
    end=END,
    bars=books,
)
print(f"最近交易日 {pd.Timestamp(latest_day):%Y-%m-%d}")
display(today if not today.empty else Markdown("_这一天没有新的买卖。_"))

snapshots = pd.concat(
    [
        position_snapshot(guarded[code], bars, strategy_name=strategy.name)
        for code, bars in books.items()
    ],
    ignore_index=True,
)
display(snapshots)

flips = window.loc[window["flag"].eq("短持翻脸")] if not window.empty else window
print(f"近 {ORDER_LOOKBACK_DAYS} 天信号 {len(window)} 条，其中短持翻脸 {len(flips)} 条")
display(window)


## 导出清单

CSV 和 Markdown 写到 `output/`。原因列保留止损和交叉说明，诊断列默认都在。连续亏损达到 `PAUSE_AFTER_LOSSES`，或短持翻脸卖出出现至少两次，会写「建议暂停加仓」，并写明当前是仍持有还是空仓，避免把提示看成新开仓。

要验证禁投拒绝：在 `orders/banned.txt` 写上清单里会出现的 6 位代码（一行一个，不要放在 `#` 后面），再跑下一格。应看到 `BannedSymbolError`，并且 `output/orders.csv` 与 `orders.md` 不会被更新。默认文件只有注释，所以这一页的默认执行会把清单写出来。这里不会发送委托。

In [ ]:
holding = {}
for code, bars in books.items():
    snap = position_snapshot(guarded[code], bars, strategy_name=strategy.name)
    if not snap.empty:
        holding[code] = str(snap.iloc[0]["state"]) == "持有"

csv_path, md_path = export_order_list(
    window,
    OUTPUT / "orders",
    pause_after_losses=PAUSE_AFTER_LOSSES,
    holding=holding,
)
print("CSV:", csv_path)
print("Markdown:", md_path)
print("不会发送任何委托。")
text = md_path.read_text(encoding="utf-8")
if PAUSE_LABEL in text:
    print(PAUSE_LABEL, "：见清单开头。提示里会写明仍持有还是空仓，不是新开仓，也不是自动停手。")
print(text)

## 登记演示成交并对照

下面用信号日之后的开盘价虚构成交，和锁定的 next_open 一致。数量按参考资金和建议仓位估算，单位是股。写出的每一笔都强制 `is_simulated=true`，不能当成实盘业绩。

佣金和印花税按上面的研究用费率写入，再从实际收益里扣掉。这不是券商价目。真实成交换成你自己的价格、数量、时间和费用，仍用 `save_fills` 写入同样的列；不要把 `simulated=True` 用在真实记录上。

对照还给出相对买入持有、以及能取到时相对沪深 300 的超额。`SPLIT_DATE` 之前是 train，当天及之后是 test。测试段收益从训练段结束时的权益接着算。

In [ ]:
parts = [
    simulate_follow_fills(bars, shown[code].signals, equity=EQUITY)
    for code, bars in books.items()
]
fills = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
fills = apply_cost_schedule(
    fills,
    commission_rate=COMMISSION_RATE,
    stamp_tax_rate=STAMP_TAX_RATE,
    min_commission=MIN_COMMISSION,
)
fill_csv, fill_json = save_fills(fills, OUTPUT / "fills", simulated=True)
print(f"演示成交 {len(fills)} 笔，is_simulated 全部为 true")
print("CSV:", fill_csv)
print("JSON:", fill_json)
try:
    benchmark = get_csi300(START, END, refresh=REFRESH, offline=OFFLINE, cache_dir=CACHE_DIR)
    print(f"基准 沪深300 {len(benchmark)} 根 · 缓存 {benchmark.attrs.get('cache_hit')}")
except Exception as exc:
    benchmark = None
    print(f"沪深300 不可用，超额只和买入持有比: {type(exc).__name__}: {exc}")
for code, bars in books.items():
    report = compare_actual(
        bars,
        guarded[code],
        fills,
        equity=EQUITY,
        benchmark=benchmark,
        split_date=SPLIT_DATE,
    )
    print(format_report(report))
    if not report.signals.empty:
        display(report.signals.head(8))

## K 线和政策与事件

红涨绿跌。三角是研究信号，短持翻脸单独标出。下方的表来自本地日历，不依赖事件约束是否打开。HTML 会再存一份到 `output/`，内嵌 plotly.js，离线也能打开。


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
for code, bars in books.items():
    label = f"{code} {NAMES.get(code, '')}".strip()
    title = f"{label} · {strategy.name} · 建议仓位 {strategy.target_weight:.0%}"
    fig = plot_signals(bars, shown[code], events=events, title=title)
    path = save_chart(fig, OUTPUT / f"{code}_{strategy.name}.html")
    print(path)
    fig.show()


## 换到均值回归

第二种策略不是双均线，默认用 20 日均线衡量偏离，不把 10 日线改短。建议仓位同样可以小于满仓。事件约束沿用上面的 `EVENT_MODE`。后面的参数表只是敏感度，跟单仍锁定 next_open，不是为了靠近每月 3%。

In [ ]:
second = MeanReversion()
print(second.name, "窗口", second.window, "偏离", second.entry_band, "建议仓位", f"{second.target_weight:.0%}")
for code, bars in books.items():
    result = EventGuard(events, mode=EVENT_MODE).apply(second.run(bars), bars)
    print(f"{code} {NAMES.get(code, '')} 信号 {len(result.signals)} 条")
    table = fill_assumption_table(
        compare_fills(bars, result),
        run_backtest(bars, result, fill="next_open_stop"),
    )
    view = table.copy()
    for column in ("total_return", "max_drawdown", "rough_monthly_return"):
        view[column] = view[column].map(lambda value: f"{value:.2%}")
    display(view[["symbol", "label", "follow", "total_return", "max_drawdown", "rough_monthly_return"]])
    fig = plot_signals(
        bars,
        apply_diagnostics(result, bars),
        events=events,
        title=f"{code} {NAMES.get(code, '')} · {second.name}",
    )
    print(save_chart(fig, OUTPUT / f"{code}_{second.name}.html"))
    fig.show()
    grid = mean_reversion_sensitivity(bars)
    print(f"{code} 均值回归敏感度（不是每月 3% 的验收）")
    display(grid)

## 事件开关样本

不拉行情。同一条仓位在 `off`、`block_entries`、`reduce` 下并排。开仓日遇到事件时，后两种都会挡住当天的新开仓。已经持有时，`block_entries` 不砍仓，`reduce` 当天降到 0。默认研究仍用 `off`。命令行同样可跑 `python scripts/event_modes.py`。


In [ ]:
for name, frame in event_switch_samples().items():
    print(name)
    display(frame)
